In [1]:
# Visual Python: Data Analysis > Import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
%matplotlib inline
#import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

In [2]:
# Visual Python: Data Analysis > Import
from sklearn.model_selection import train_test_split
from sklearn import metrics

In [3]:
# Visual Python: Data Analysis > File
df = pd.read_csv('fraudTrain.csv')
df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786.0,-11.881118,10.071785,-9.834783,-2.066656,-5.364473,-2.606837,-4.918215,7.305334,1.914428,...,0.213454,0.111864,1.014480,-0.509348,1.436807,0.250034,0.943651,0.823731,0.77,0
284803,172787.0,-0.732789,-0.055080,2.035030,-0.738589,0.868229,1.058415,0.024330,0.294869,0.584800,...,0.214205,0.924384,0.012463,-1.016226,-0.606624,-0.395255,0.068472,-0.053527,24.79,0
284804,172788.0,1.919565,-0.301254,-3.249640,-0.557828,2.630515,3.031260,-0.296827,0.708417,0.432454,...,0.232045,0.578229,-0.037501,0.640134,0.265745,-0.087371,0.004455,-0.026561,67.88,0
284805,172788.0,-0.240440,0.530483,0.702510,0.689799,-0.377961,0.623708,-0.686180,0.679145,0.392087,...,0.265245,0.800049,-0.163298,0.123205,-0.569159,0.546668,0.108821,0.104533,10.00,0


In [4]:
df['Class'].value_counts()

Class
0    284315
1       492
Name: count, dtype: int64

In [5]:
# A look at the shape of the data
original_rows = len(df)
df.shape

(284807, 31)

In [6]:
# Removing duplicate values 
df.drop_duplicates(subset = None, keep = "first", inplace = True, ignore_index = True)

In [7]:
# Again having a look at the shape of the data to check the number of removed rows
dedup_rows = len(df)
df.shape

(283726, 31)

In [8]:
# Total rows removed 
print("Total duplicate rows removed : ", original_rows -dedup_rows)

Total duplicate rows removed :  1081


In [9]:
df["Class"].value_counts()

Class
0    283253
1       473
Name: count, dtype: int64

In [10]:
%%time
import pandas as pd
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
from cryptography.hazmat.primitives import padding
import os
import base64

# Function to encrypt a column
def encrypt_column(column, key):
    # Convert column to a single string
    column_string = ','.join(map(str, column.values)).encode()
    
    # Generate a random 16-byte IV
    iv = os.urandom(16)
    
    # Create AES cipher in CBC mode
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    encryptor = cipher.encryptor()
    
    # Pad the data to be a multiple of the AES block size
    padder = padding.PKCS7(128).padder()
    padded_data = padder.update(column_string) + padder.finalize()
    
    # Encrypt the padded data
    ciphertext = encryptor.update(padded_data) + encryptor.finalize()
    
    return iv + ciphertext  # Prepend the IV for use in decryption

# Function to decrypt a column
def decrypt_column(encrypted_column, key):
    # Extract the IV (first 16 bytes)
    iv = encrypted_column[:16]
    ciphertext = encrypted_column[16:]
    
    # Create AES cipher in CBC mode
    cipher = Cipher(algorithms.AES(key), modes.CBC(iv), backend=default_backend())
    decryptor = cipher.decryptor()
    
    # Decrypt and then unpad the data
    padded_data = decryptor.update(ciphertext) + decryptor.finalize()
    unpadder = padding.PKCS7(128).unpadder()
    original_data = unpadder.update(padded_data) + unpadder.finalize()
    
    # Convert back to a list of values
    return original_data.decode().split(',')

# Main Code
key = os.urandom(32)  # AES-256 key

# Load dataset
df = pd.read_csv("fraudTrain.csv")  # Replace with your dataset path

# Encrypt each column
encrypted_columns = {
    column: encrypt_column(df[column], key) for column in df.columns
}

# Create a DataFrame to store encrypted data
encrypted_df = pd.DataFrame(
    {
        column: [base64.b64encode(encrypted_columns[column]).decode()]
        for column in df.columns
    }
).transpose()
encrypted_df.columns = ["encrypted_data"]

# Decrypt each column
decrypted_columns = {
    column: decrypt_column(base64.b64decode(encrypted_df.loc[column, "encrypted_data"]), key)
    for column in df.columns
}

# Convert decrypted columns back to a DataFrame
decrypted_df = pd.DataFrame(decrypted_columns)

# Verify results
print("Original DataFrame:")
print(df.head())
print("\nEncrypted DataFrame (Column-wise):")
print(encrypted_df)
print("\nDecrypted DataFrame:")
print(decrypted_df.head())


Original DataFrame:
   Time        V1        V2        V3        V4        V5        V6        V7  \
0   0.0 -1.359807 -0.072781  2.536347  1.378155 -0.338321  0.462388  0.239599   
1   0.0  1.191857  0.266151  0.166480  0.448154  0.060018 -0.082361 -0.078803   
2   1.0 -1.358354 -1.340163  1.773209  0.379780 -0.503198  1.800499  0.791461   
3   1.0 -0.966272 -0.185226  1.792993 -0.863291 -0.010309  1.247203  0.237609   
4   2.0 -1.158233  0.877737  1.548718  0.403034 -0.407193  0.095921  0.592941   

         V8        V9  ...       V21       V22       V23       V24       V25  \
0  0.098698  0.363787  ... -0.018307  0.277838 -0.110474  0.066928  0.128539   
1  0.085102 -0.255425  ... -0.225775 -0.638672  0.101288 -0.339846  0.167170   
2  0.247676 -1.514654  ...  0.247998  0.771679  0.909412 -0.689281 -0.327642   
3  0.377436 -1.387024  ... -0.108300  0.005274 -0.190321 -1.175575  0.647376   
4 -0.270533  0.817739  ... -0.009431  0.798278 -0.137458  0.141267 -0.206010   

        V26 